In [ ]:

import torch
import os, sys
import polars as pl


sys.path.insert(0, os.path.abspath(".."))


MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

# Data Preprocess

## 1. Interpolate part's sequence

### Goal
- 자재(`oper_part_no`)별로 주차(`demand_dt = yyyyww`) 시계열이 **중간에 누락된 주차가 존재**할 수 있음
- 누락 구간을 **주차 단위로 연속 인덱스(weekly index)** 로 보간(interpolation)하여,
  - 모든 자재가 **동일한 규칙(매주 1개 timestep)** 을 갖도록 정규화
  - 이후 모델 학습/분석에서 **seq(순번), 누락주 처리, 0-demand 구간 처리**를 일관되게 수행 가능하게 함

---

### Methodology

- `demand_dt(yyyyww)` 를 **ISO week 기반 Monday(date)** 로 변환
  - `yyyyww_to_monday(yyyyww)` 사용
  - `date.fromisocalendar(year, week, 1)` → 해당 주차의 월요일 반환
- 자재별(`oper_part_no`)로 관측된 `demand_date`의 **min~max 구간**을 계산
- 각 자재별 min~max 사이에 대해 **주 단위(1w) date_range 생성**
  - `pl.date_range(min_date, max_date, interval='1w', closed='both')`
- 생성된 full weekly index를 explode 하여 **(oper_part_no, demand_date)** 레벨의 완전한 시계열 인덱스 테이블 구성
- 원본 데이터(df2)와 full index를 **left join** 하여 누락 주차를 포함한 전체 시계열 구성
  - 누락된 `demand_qty`는 `0.0`으로 fill
- `demand_date(Monday)` 를 다시 `yyyyww` 로 역변환
  - `monday_to_yyyyww(date)` 사용
  - `d.isocalendar()` 기반으로 `year*100 + week`
- 자재별로 누적 순번 `seq` 생성
  - `pl.cum_count('oper_part_no').over('oper_part_no')`
  - 각 자재 내에서 `0..T-1` 형태의 timestep index 역할
---

In [ ]:
from datetime import date
def yyyyww_to_monday(yyyyww: int) -> date:
    y = yyyyww // 100
    w = yyyyww % 100
    return date.fromisocalendar(y, w, 1)

def monday_to_yyyyww(d: date) -> int:
    iso = d.isocalendar()
    return int(iso.year) * 100 + int(iso.week)


df = (pl
    .read_parquet(DIR + 'sample_data/target_dyn_demand.parquet')
    .sort(['oper_part_no', 'demand_dt'])
 )

df2 = (
    df.with_columns(
        pl.col('demand_dt')
          .map_elements(yyyyww_to_monday, return_dtype = pl.Date)
          .alias('demand_date')
    )
)

full_index = (
    df2.group_by('oper_part_no')
       .agg(
            pl.date_range(
                pl.col('demand_date').min(),
                pl.col('demand_date').max(),
                interval = '1w',
                closed = 'both',
                eager = False,
            ).alias('demand_date')
    )
    .explode('demand_date')
)

out = (
    full_index
        .join(df2.select(['oper_part_no', 'demand_date', 'demand_qty']),
              on = ['oper_part_no', 'demand_date'],
              how = 'left'
              )
        .with_columns(
            pl.col('demand_qty').fill_null(0.0),
            pl.col('demand_date')
              .map_elements(monday_to_yyyyww, return_dtype = pl.Int64)
              .alias('demand_dt'),
            pl.cum_count('oper_part_no').over('oper_part_no').alias('seq')
        )
        .select(['oper_part_no', 'demand_dt', 'demand_qty', 'seq'])
        .sort(['oper_part_no', 'demand_dt'])
)

out

## Select Intermittent Parts using intermittent detector.

In [ ]:
from utils.intermittent_detector import IntermittentConfig, IntermittentDetector

intermittent_config = IntermittentConfig(
    id_col = 'oper_part_no',
    date_col = 'demand_dt',
    y_col = 'demand_qty',
    seq_col = 'seq',
    adi_threshold = 1.32,
    cv2_threshold = 0.49,
    epsilon = 0.0,
    min_periods = 10,
    no_demand_label = 'no_demand',
    insufficient_label = 'insufficient'
)

result = IntermittentDetector(
    freq = 'weekly',
    config = intermittent_config,
).run(df = out, return_stats = True )

result

## Filter Intermittent parts & See Plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

intermittent_parts = result.filter(pl.col('demand_type') == 'intermittent').select('oper_part_no').unique()
intermittent_df = out.join(intermittent_parts, on = 'oper_part_no', how = 'right')

pdf = intermittent_df.to_pandas()
pdf = pdf.sort_values(['oper_part_no', 'seq'])
fig, ax = plt.subplots(figsize = (18, 7))
for part, g in pdf.groupby('oper_part_no', sort = False):
    ax.plot(g['seq'].values, g['demand_qty'].values, linewidth = 1, alpha = 0.25)

ax.set_title('Demand Qty by oper_part_no (all parts)')
ax.set_xlabel('seq')
ax.set_ylabel('demand_qty')
ax.grid(True, alpha = 0.3)

plt.tight_layout()
plt.show()

## Making Marker DataFrame

In [ ]:
def make_marks_intermittent(df: pl.DataFrame, K: int = 12, p1: float = 0.99, p2: float = 0.999) -> pl.DataFrame:
    assert K >= 6

    # event 생성
    ev = (
        df.filter(pl.col('demand_qty') > 0) # 0 이상인것들만
          .sort(['oper_part_no', 'seq'])
          .with_columns([
              pl.col('demand_qty').log1p().alias('z'),
             (pl.col('seq') - pl.col('seq').shift(1).over('oper_part_no'))
                .fill_null(0)
                .cast(pl.Int32)
                .alias('delta_t'),
        ])
    )

    q1 = ev.select(pl.col("demand_qty").quantile(p1, "nearest").alias("q1")).item()
    q2 = ev.select(pl.col("demand_qty").quantile(p2, "nearest").alias("q2")).item()
    base = ev.filter(pl.col('demand_qty') <= q1)

    n_base_bins = K - 2
    cut_probs = [i / n_base_bins for i in range(1, n_base_bins)]

    q_expr = [
        pl.col("z").quantile(p, "nearest").alias(f"q_{i:02d}")
        for i, p in enumerate(cut_probs, start=1)
    ]

    # base가 너무 작거나(이벤트 부족) cuts가 비는 경우를 대비
    if base.height == 0:
        cuts = []
    else:
        row = base.select(q_expr).row(0)
        cuts = sorted(set(float(x) for x in row if x is not None))\

    # z -> base_bin (0..n_base_bins-1)
    expr = None
    for i, c in enumerate(cuts):
        if expr is None:
            expr = pl.when(pl.col('z') <= c).then(i)
        else:
            expr = expr.when(pl.col('z') <= c).then(i)

    base_bin = (pl.lit(0) if expr is None else expr.otherwise(len(cuts))).clip(0, n_base_bins - 1)

    out = ev.with_columns(
        pl.when(pl.col('demand_qty') <= q1)
          .then(base_bin)
          .when(pl.col('demand_qty') <= q2)
          .then(pl.lit(K-2))
          .otherwise(pl.lit(K-1))
          .cast(pl.Int32)
          .alias('mark')
    )

    return out.select(['oper_part_no', 'demand_dt', 'seq', 'delta_t', 'demand_qty', 'z', 'mark'])

marked_df = make_marks_intermittent(intermittent_df, K=8, p1=0.99, p2=0.999)

marked_df.group_by("mark").len().sort("mark")
marked_df.select(pl.col("delta_t")).describe()

In [ ]:
# marked_df.write_parquet(DIR + 'sample_data/marked_target_df.parquet')
marked_df.schema